Event-based Directional Prediction on SPY using Internal Data & ML Libraries

This notebook executes an event-based pipeline on SPY:
- Fetch data
- Construct dollar bars
- Detect events (CUSUM)
- Label with Triple Barrier
- Build compact features
- Train a baseline classifier with time-based splits
- Convert predictions to weights and compute returns
- Run a single final backtest call

Intermediate outputs are shown for readability.

Data sources used

- Primary market data: SPY OHLCV daily bars fetched via findata.equity_prices.get_equity_prices over the parameterized window START_DATE to END_DATE.
- All subsequent datasets are derived within this notebook from that OHLCV:
  - Approximate dollar bars: research_bars.build_dollar_bars_from_ohlcv
  - Volatility estimates: mlfinlab.util.volatility.get_daily_vol
  - Event detection (CUSUM): mlfinlab.filters.filters.cusum_filter
  - Triple-barrier events and labels: mlfinlab.labeling.labeling.get_events / mlfinlab.labeling.labeling.get_bins
  - Model feature set: research_ml.build_features_basic
  - Predictions and time-split evaluation: research_ml.train_eval_time_split
  - Portfolio construction and returns series: portfolio.* helpers

No other external data vendors or alternative datasets are used in this notebook.

In [1]:
# Imports and settings
import pandas as pd
import numpy as np

from findata.equity_prices import get_equity_prices

from mlfinlab.util.volatility import get_daily_vol
from mlfinlab.filters.filters import cusum_filter
from mlfinlab.labeling.labeling import get_events, get_bins

from typing import Tuple, Dict

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 50)

# Utility to peek at DataFrames compactly
def peek(df, n=5, name=None):
    if name:
        print(f"\n>>> {name}: shape={getattr(df, 'shape', None)} index_range=({df.index.min()} -> {df.index.max()})")
    display(df.head(n))

# -----------------------------
# Minimal helper: approximate dollar bars from daily OHLCV
# -----------------------------
def build_dollar_bars_from_ohlcv(ohlcv: pd.DataFrame,
                                 dollar_value_per_bar: str = "median_20d",
                                 min_obs_per_bar: int = 1,
                                 debug: bool = True) -> pd.DataFrame:
    df = ohlcv.copy()
    if isinstance(df.columns, pd.MultiIndex):
        if df.columns.nlevels == 2 and len(df.columns.levels[1]) == 1:
            df = pd.concat({c: df[(c, df.columns.levels[1][0])] for c in df.columns.levels[0]}, axis=1)
            df.columns = list(df.columns)
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        if col not in df.columns:
            raise ValueError(f"Missing column {col} in OHLCV")
    dv = (df["Close"] * df["Volume"]).rename("DollarValue")
    if dollar_value_per_bar == "median_20d":
        threshold = dv.rolling(20, min_periods=1).median()
    else:
        try:
            thr = float(dollar_value_per_bar)
        except Exception:
            thr = dv.median()
        threshold = pd.Series(thr, index=dv.index)
    bars = []
    cum_dv = 0.0
    bar_open = bar_high = bar_low = bar_close = None
    bar_vol = 0.0
    count_obs = 0
    for ts, row in df.iterrows():
        price_o, price_h, price_l, price_c, vol = row[["Open", "High", "Low", "Close", "Volume"]]
        if bar_open is None:
            bar_open = price_o
            bar_high = price_h
            bar_low = price_l
            bar_vol = 0.0
        else:
            bar_high = max(bar_high, price_h)
            bar_low = min(bar_low, price_l)
        bar_close = price_c
        bar_vol += vol
        cum_dv += price_c * vol
        count_obs += 1
        thr_now = threshold.loc[ts]
        if (count_obs >= min_obs_per_bar) and (cum_dv >= thr_now):
            bars.append({
                'DateTime': ts,
                'Open': bar_open,
                'High': bar_high,
                'Low': bar_low,
                'Close': bar_close,
                'Volume': bar_vol,
                'DollarValue': cum_dv,
            })
            cum_dv = 0.0
            bar_open = bar_high = bar_low = bar_close = None
            bar_vol = 0.0
            count_obs = 0
    bars_df = pd.DataFrame(bars)
    if bars_df.empty:
        if debug:
            print("Warning: No dollar bars formed; returning original OHLCV copy")
        out = df.copy()
        out["DollarValue"] = dv
        return out
    bars_df = bars_df.set_index('DateTime').sort_index()
    return bars_df

# -----------------------------
# Minimal helper: build compact features
# -----------------------------

def _rsi(series: pd.Series, window: int = 14) -> pd.Series:
    delta = series.diff()
    up = delta.clip(lower=0)
    down = -delta.clip(upper=0)
    roll_up = up.ewm(alpha=1/window, adjust=False).mean()
    roll_down = down.ewm(alpha=1/window, adjust=False).mean()
    rs = roll_up / (roll_down.replace(0, np.nan))
    rsi = 100 - (100 / (1 + rs))
    return rsi


def build_features_basic(ohlcv: pd.DataFrame,
                         labels: pd.DataFrame,
                         feature_windows: list,
                         include: list,
                         drop_na: bool = True,
                         debug: bool = True) -> Tuple[pd.DataFrame, pd.Series, Dict]:
    df = ohlcv.copy()
    close = df['Close']
    ret = close.pct_change()
    feats = pd.DataFrame(index=df.index)
    if 'ret_lag1' in include:
        feats['ret_lag1'] = ret.shift(1)
    if 'ret_lag5' in include:
        feats['ret_lag5'] = ret.rolling(5).sum().shift(1)
    if 'vol_ewm' in include:
        feats['vol_ewm'] = ret.ewm(span=20, adjust=False).std().shift(1)
    if 'zscore_20' in include:
        mu = ret.rolling(20).mean()
        sd = ret.rolling(20).std()
        feats['zscore_20'] = ((ret - mu) / sd).shift(1)
    if 'rsi_14' in include:
        feats['rsi_14'] = _rsi(close, 14).shift(1) / 100.0
    y = labels['bin'].reindex(feats.index)
    X = feats
    if drop_na:
        mask = X.notna().all(axis=1) & y.notna()
        X = X[mask]
        y = y[mask]
    meta = {"feature_windows": feature_windows, "included": include}
    if debug:
        print(f"Built features: {list(X.columns)}; rows={len(X)}; label balance: {(y==1).mean():.3f} long fraction")
    return X, y.astype(int), meta

# -----------------------------
# Minimal helper: time-based train/test splits with baseline classifier
# -----------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

def train_eval_time_split(X: pd.DataFrame,
                          y: pd.Series,
                          index: pd.Index,
                          model_type: str = 'logistic_regression',
                          time_splits: int = 5,
                          test_size_fraction: float = 0.2,
                          standardize: bool = True,
                          class_weight: str | None = 'balanced',
                          random_state: int = 42,
                          debug: bool = True):
    if model_type != 'logistic_regression':
        raise ValueError('Only logistic_regression is implemented in this minimal helper')
    if standardize:
        model = make_pipeline(StandardScaler(with_mean=False), LogisticRegression(max_iter=1000, class_weight=class_weight, random_state=random_state))
    else:
        model = LogisticRegression(max_iter=1000, class_weight=class_weight, random_state=random_state)

    n = len(X)
    test_size = max(1, int(n * test_size_fraction))
    split_points = []
    for i in range(time_splits, 0, -1):
        end = n - (i - 1) * test_size
        start = max(1, end - test_size)
        if end <= start or end > n:
            continue
        split_points.append((slice(0, start - 1), slice(start - 1, end)))
    uniq = []
    for tr, te in split_points:
        if (tr, te) not in uniq:
            uniq.append((tr, te))
    split_points = uniq

    y_pred_all = pd.Series(index=X.index, dtype=float)
    y_proba_all = pd.Series(index=X.index, dtype=float)
    metrics: Dict[str, Dict] = {}
    used_splits = 0

    for k, (tr, te) in enumerate(split_points, start=1):
        X_tr, y_tr = X.iloc[tr], y.iloc[tr]
        X_te, y_te = X.iloc[te], y.iloc[te]
        # Skip if training set has < 2 classes
        if y_tr.nunique() < 2 or len(X_te) == 0:
            if debug:
                print(f"Split {k}: skipped (insufficient class diversity or empty test)")
            continue
        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_te)[:, 1]
        pred = np.where(proba >= 0.5, 1, -1)
        y_proba_all.iloc[te] = proba
        y_pred_all.iloc[te] = pred
        try:
            auc = roc_auc_score((y_te==1).astype(int), proba)
        except ValueError:
            auc = np.nan
        acc = accuracy_score(y_te, pred)
        f1 = f1_score((y_te==1).astype(int), (pred==1).astype(int), zero_division=0)
        metrics[f"split_{k}"] = {"auc": auc, "accuracy": acc, "f1_long": f1}
        used_splits += 1
        if debug:
            print(f"Split {k}: AUC={auc:.3f}, ACC={acc:.3f}, F1_long={f1:.3f}")

    if used_splits == 0:
        raise ValueError("No valid time splits with at least two classes in training. Try adjusting labeling or parameters.")

    accs = [v['accuracy'] for v in metrics.values() if not pd.isna(v['accuracy'])]
    aucs = [v['auc'] for v in metrics.values() if not pd.isna(v['auc'])]
    f1s = [v['f1_long'] for v in metrics.values() if not pd.isna(v['f1_long'])]
    metrics['summary'] = {
        'accuracy_mean': float(np.mean(accs)) if accs else np.nan,
        'auc_mean': float(np.mean(aucs)) if aucs else np.nan,
        'f1_long_mean': float(np.mean(f1s)) if f1s else np.nan,
        'n_splits': used_splits
    }

    y_pred_all = y_pred_all.dropna()
    y_proba_all = y_proba_all.dropna()
    return metrics, y_proba_all, y_pred_all

# -----------------------------
# Minimal helpers: portfolio weights & returns
# -----------------------------

def make_weights_from_predictions(pred_label: pd.Series,
                                  pred_proba: pd.Series,
                                  long_short: bool = True,
                                  prob_threshold: float = 0.5,
                                  max_leverage: float = 1.0,
                                  align_to_index: pd.Index | None = None,
                                  debug: bool = True) -> pd.DataFrame:
    idx = align_to_index if align_to_index is not None else pred_label.index
    w = pd.Series(0.0, index=idx)
    pl = pred_label.reindex(idx).fillna(0)
    if long_short:
        w = pl.clip(-1, 1).astype(float)
    else:
        if pred_proba is None:
            raise ValueError('pred_proba required for long/flat mode')
        pp = pred_proba.reindex(idx).fillna(0.0)
        w = (pp >= prob_threshold).astype(float)
    w = (w * max_leverage).to_frame('SPY')
    if debug:
        print(f"Weights built: long_short={long_short}, mean |w|={w['SPY'].abs().mean():.3f}")
    return w


def compute_returns_from_weights(weights: pd.DataFrame,
                                 prices_close: pd.Series,
                                 shift_weights_by_1: bool = True,
                                 gross_leverage_cap: float = 1.0,
                                 debug: bool = True) -> pd.DataFrame:
    px = prices_close.reindex(weights.index).dropna()
    w = weights.reindex(px.index).fillna(0.0).copy()
    if shift_weights_by_1:
        w = w.shift(1).fillna(0.0)
    w['SPY'] = w['SPY'].clip(-gross_leverage_cap, gross_leverage_cap)
    ret = px.pct_change().fillna(0.0)
    port_ret = (w['SPY'] * ret).to_frame('SPY')
    if debug:
        print(f"Mean daily return: {port_ret['SPY'].mean():.6f}")
    return port_ret

Task 1 — Fetch SPY OHLCV daily bars (findata.get_equity_prices)

In [2]:
# Parameters
START_DATE = "2015-01-01"
END_DATE   = "2025-01-01"

# Execute task: fetch_prices
prices_ohlcv = get_equity_prices(
    tickers=["SPY"],
    start_date=START_DATE,
    end_date=END_DATE,
    fields=["Open", "High", "Low", "Close", "Volume"],
    frequency="1d",
    auto_adjust=True,
)

peek(prices_ohlcv, name="prices_ohlcv (daily OHLCV)")


>>> prices_ohlcv (daily OHLCV): shape=(2516, 5) index_range=(2015-01-02 00:00:00 -> 2024-12-31 00:00:00)


Price,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,170.125000,171.325815,169.089824,170.911744,121465900
2015-01-05,167.052612,169.247181,166.746204,169.081555,169632600
2015-01-06,165.479111,167.880714,164.684090,167.358981,209151400
2015-01-07,167.541229,167.880770,166.356994,166.804184,125346700
2015-01-08,170.514221,170.729546,168.932482,168.949035,147217800


Task 2 — Construct approximate dollar bars from daily OHLCV (research_bars.build_dollar_bars_from_ohlcv)

In [3]:
# Execute task: build_dollar_bars
# Parameters: dollar_value_per_bar='median_20d', min_obs_per_bar=1, debug=True

dollar_bars = build_dollar_bars_from_ohlcv(
    ohlcv=prices_ohlcv,
    dollar_value_per_bar="median_20d",
    min_obs_per_bar=1,
    debug=True,
)

peek(dollar_bars, name="dollar_bars (approx from daily OHLCV)")
print("Columns:", list(dollar_bars.columns))


>>> dollar_bars (approx from daily OHLCV): shape=(1732, 6) index_range=(2015-01-02 00:00:00 -> 2024-12-31 00:00:00)

,Open,High,Low,Close,Volume,DollarValue
DateTime,,,,,,
2015-01-02,170.911744,171.325815,169.089824,170.125000,121465900.0,2.066439e+10
2015-01-05,169.081555,169.247181,166.746204,167.052612,169632600.0,2.833757e+10
2015-01-06,167.358981,167.880714,164.684090,165.479111,209151400.0,3.461019e+10
2015-01-08,166.804184,170.729546,166.356994,170.514221,272564500.0,4.610347e+10
2015-01-09,170.928295,170.944861,168.534968,169.147797,158567300.0,2.682131e+10


Columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'DollarValue']


Task 3 — Estimate daily volatility for thresholds/targets (mlfinlab.util.volatility.get_daily_vol)

In [4]:
# Execute task: daily_vol
close_series = dollar_bars["Close"].dropna()
vol_series = get_daily_vol(close=close_series, lookback=100)

peek(vol_series.to_frame("vol"), name="vol_series (daily vol)")


>>> vol_series (daily vol): shape=(1731, 1) index_range=(2015-01-05 00:00:00 -> 2024-12-31 00:00:00)


,vol
DateTime,
2015-01-05,NaN
2015-01-06,0.006540
2015-01-08,0.031185
2015-01-09,0.028728
2015-01-13,0.025435


Task 4 — Detect events via symmetric CUSUM on dollar-bar closes (mlfinlab.filters.filters.cusum_filter)

In [5]:
# Execute task: cusum_events
# Threshold set as vol_series * 1.0 per task spec
# Align threshold index to price series
threshold = vol_series.reindex(close_series.index).fillna(method='ffill') * 1.0

# cusum_filter returns event timestamps
t_events = cusum_filter(raw_time_series=close_series, threshold=threshold, time_stamps=True)

print(f"Detected {len(t_events)} events via CUSUM")
print("First 10 event timestamps:")
print(pd.DatetimeIndex(t_events)[:10])

Detected 1718 events via CUSUM
First 10 event timestamps:
DatetimeIndex(['2015-01-06', '2015-01-08', '2015-01-09', '2015-01-13', '2015-01-14', '2015-01-15', '2015-01-16', '2015-01-21', '2015-01-22', '2015-01-26'], dtype='datetime64[ns]', freq=None)


/var/folders/jr/prby656n267_0pztcwybgy5r0000gn/T/ipykernel_90584/1866813802.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  threshold = vol_series.reindex(close_series.index).fillna(method='ffill') * 1.0


Task 5 — Apply triple barrier to get event spans and targets (mlfinlab.labeling.labeling.get_events)

In [6]:
# Execute task: triple_barrier_events
# pt_sl = [1.0, 1.0]; min_ret = 0.001; vertical_barrier_times=False; side_prediction=None; verbose=True

pt_sl = [1.0, 1.0]
min_ret = 0.001

tb_events = get_events(
    close=close_series,
    t_events=pd.DatetimeIndex(t_events),
    pt_sl=pt_sl,
    target=vol_series,
    min_ret=min_ret,
    num_threads=4,
    vertical_barrier_times=False,
    side_prediction=None,
    verbose=True,
)

peek(tb_events, name="tb_events (triple barrier events)")

1/4 apply_pt_sl_on_t1 done after 0.02 minutes. 0.07 minutes remaining.
2/4 apply_pt_sl_on_t1 done after 0.02 minutes. 0.02 minutes remaining.
3/4 apply_pt_sl_on_t1 done after 0.02 minutes. 0.01 minutes remaining.
4/4 apply_pt_sl_on_t1 done after 0.02 minutes. done.



>>> tb_events (triple barrier events): shape=(1718, 4) index_range=(2015-01-06 00:00:00 -> 2024-12-31 00:00:00)


,t1,trgt,pt,sl
2015-01-06,2015-01-08,0.006540,1.0,1.0
2015-01-08,2015-01-15,0.031185,1.0,1.0
2015-01-09,2015-02-18,0.028728,1.0,1.0
2015-01-13,2015-02-13,0.025435,1.0,1.0
2015-01-14,2015-01-22,0.023615,1.0,1.0


Task 6 — Create direction labels from triple-barrier events (mlfinlab.labeling.labeling.get_bins)

In [7]:
# Execute task: make_labels
labels = get_bins(triple_barrier_events=tb_events, close=close_series)

# Basic sanity: bin distribution
label_counts = labels['bin'].value_counts(dropna=False)
print("Label distribution (bin):")
print(label_counts)

peek(labels, name="labels (first bins)")

Label distribution (bin):
bin
 1.0    1022
-1.0     694
Name: count, dtype: int64

>>> labels (first bins): shape=(1716, 2) index_range=(2015-01-06 00:00:00 -> 2024-12-27 00:00:00)


,ret,bin
2015-01-06,0.030427,1.0
2015-01-08,-0.033414,-1.0
2015-01-09,0.028788,1.0
2015-01-13,0.038104,1.0
2015-01-14,0.026088,1.0


Task 7 — Build compact feature set on dollar bars (research_ml.build_features_basic)

In [8]:
# Execute task: build_features
X, y, meta = build_features_basic(
    ohlcv=dollar_bars,
    labels=labels,
    feature_windows=[5, 10, 20],
    include=["ret_lag1", "ret_lag5", "vol_ewm", "zscore_20", "rsi_14"],
    drop_na=True,
    debug=True,
)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
peek(X, name="X (features)")
peek(y.to_frame('y'), name="y (labels aligned)")

Built features: ['ret_lag1', 'ret_lag5', 'vol_ewm', 'zscore_20', 'rsi_14']; rows=1697; label balance: 0.594 long fraction
Features shape: (1697, 5)
Target shape: (1697,)

>>> X (features): shape=(1697, 5) index_range=(2015-02-20 00:00:00 -> 2024-12-27 00:00:00)


,ret_lag1,ret_lag5,vol_ewm,zscore_20,rsi_14
DateTime,,,,,
2015-02-20,0.001668,0.025701,0.014789,0.029468,0.419308
2015-02-24,0.005283,0.024735,0.014120,0.205609,0.437350
2015-02-26,0.002698,0.034662,0.013421,-0.022483,0.446854
2015-02-27,-0.002030,0.021392,0.012780,-0.281006,0.440805
2015-03-03,-0.003406,0.004212,0.012205,-0.418206,0.430301



>>> y (labels aligned): shape=(1697, 1) index_range=(2015-02-20 00:00:00 -> 2024-12-27 00:00:00)


,y
DateTime,
2015-02-20,-1
2015-02-24,-1
2015-02-26,-1
2015-02-27,-1
2015-03-03,-1


Task 8 — Train baseline classifier and evaluate with time-based splits (research_ml.train_eval_time_split)

In [9]:
# Execute task: train_eval
cv_metrics, pred_proba, pred_label = train_eval_time_split(
    X=X,
    y=y,
    index=X.index,
    model_type="logistic_regression",
    time_splits=5,
    test_size_fraction=0.2,
    standardize=True,
    class_weight="balanced",
    random_state=42,
    debug=True,
)

print("Cross-validated metrics (summary):")
for k, v in cv_metrics.items():
    print(f"  {k}: {v}")

peek(pred_label.to_frame('pred_label'), name="Out-of-sample predicted labels")
peek(pred_proba.to_frame('pred_proba'), name="Out-of-sample predicted probas")

Split 1: skipped (insufficient class diversity or empty test)
Split 2: AUC=0.518, ACC=0.550, F1_long=0.648
Split 3: AUC=0.435, ACC=0.529, F1_long=0.646
Split 4: AUC=0.413, ACC=0.412, F1_long=0.435
Split 5: AUC=0.526, ACC=0.515, F1_long=0.569
Cross-validated metrics (summary):
  split_2: {'auc': 0.5178904140841858, 'accuracy': 0.55, 'f1_long': 0.6482758620689655}
  split_3: {'auc': 0.43453902842532716, 'accuracy': 0.5294117647058824, 'f1_long': 0.6460176991150443}
  split_4: {'auc': 0.4126890727216087, 'accuracy': 0.4117647058823529, 'f1_long': 0.4350282485875706}
  split_5: {'auc': 0.5257502913752914, 'accuracy': 0.5147058823529411, 'f1_long': 0.5691906005221932}
  summary: {'accuracy_mean': 0.5014705882352941, 'auc_mean': 0.47271720165160325, 'f1_long_mean': 0.5746281025734433, 'n_splits': 4}

>>> Out-of-sample predicted labels: shape=(1357, 1) index_range=(2017-02-17 00:00:00 -> 2024-12-27 00:00:00)


,pred_label
DateTime,
2017-02-17,1.0
2017-02-21,1.0
2017-02-23,1.0
2017-02-24,1.0
2017-02-28,1.0



>>> Out-of-sample predicted probas: shape=(1357, 1) index_range=(2017-02-17 00:00:00 -> 2024-12-27 00:00:00)


,pred_proba
DateTime,
2017-02-17,0.565677
2017-02-21,0.555789
2017-02-23,0.528437
2017-02-24,0.592105
2017-02-28,0.587954


Task 9 — Convert predictions to SPY weights (directional, unit leverage) (portfolio.make_weights_from_predictions)

In [10]:
# Execute task: portfolio_weights
asset_weights = make_weights_from_predictions(
    pred_label=pred_label,
    pred_proba=pred_proba,
    long_short=True,
    prob_threshold=0.5,
    max_leverage=1.0,
    align_to_index=dollar_bars.index,
    debug=True,
)

peek(asset_weights, name="asset_weights (SPY)")

Weights built: long_short=True, mean |w|=0.783

>>> asset_weights (SPY): shape=(1732, 1) index_range=(2015-01-02 00:00:00 -> 2024-12-31 00:00:00)


,SPY
DateTime,
2015-01-02,0.0
2015-01-05,0.0
2015-01-06,0.0
2015-01-08,0.0
2015-01-09,0.0


Task 10 — Compute portfolio returns using SPY close-to-close returns (portfolio.compute_returns_from_weights)

In [11]:
# Execute task: portfolio_returns
asset_returns = compute_returns_from_weights(
    weights=asset_weights,
    prices_close=close_series,
    shift_weights_by_1=True,
    gross_leverage_cap=1.0,
    debug=True,
)

peek(asset_returns, name="asset_returns (SPY)")

Mean daily return: -0.000364

>>> asset_returns (SPY): shape=(1732, 1) index_range=(2015-01-02 00:00:00 -> 2024-12-31 00:00:00)


,SPY
DateTime,
2015-01-02,0.0
2015-01-05,-0.0
2015-01-06,-0.0
2015-01-08,0.0
2015-01-09,-0.0


Final — Run user-defined backtest exactly once (consumes asset_weights and asset_returns)

In [12]:
# The environment is expected to provide `user_backtest(asset_weights: pd.DataFrame, asset_returns: pd.DataFrame)`
# This call is required exactly once per the task specification.
try:
    backtest_result = user_backtest(asset_weights=asset_weights, asset_returns=asset_returns)
    print("Backtest finished. Summary:")
    print(backtest_result)
except NameError:
    print("user_backtest function is not defined in the environment. Please define it and re-run this cell.")

user_backtest function is not defined in the environment. Please define it and re-run this cell.
